# 2. Change detection by segmentation and comparison

The most direct way to find new buildings is to find all of them in both steps, then compare the predictions. To this end, we will need a building segmentor.

TorchGeo does not have a trained version, but it has models and datasets to train one (and while making this tutorial I really think I should probs some pretrained one for this). [LandCover.ai](https://landcover.ai.linuxpolska.com/) has aerial orthophotos of Poland with five classes: background, building, woodland,
water and road. I trained a U-Net on it before the workshop, so we will use that here for a small finetuning step on our dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from lightning import Trainer
from lightning.pytorch.callbacks import TQDMProgressBar
from matplotlib.colors import ListedColormap
from shapely.geometry import box
from torch.utils.data import DataLoader
from torchmetrics.classification import BinaryF1Score, BinaryJaccardIndex

from torchgeo.datasets import RasterDataset, VectorDataset
from torchgeo.samplers import GriddedPatchSampler, RandomPatchSampler
from torchgeo.tasks import SemanticSegmentation

PREPARED = Path('data/prepared')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

train_roi = box(457250, 100000, 458250, 103000)
test_roi = box(458250, 100000, 458750, 103000)
val_roi = box(458750, 100000, 459500, 103000)

## Data

We use the data we prepared before. The model expects the images to be normalised to interval [0, 1] so we will also add a small tranform to handle this. These transforms can in general be quite diverse and I really urge you to think about this when doing your own things as these are really important. In first case, if model expect certain range we MUST match our data to that (simple standardisation is usually what we do). On the other hand, things like flips, rotations, crops serve as "data expansion", virtutally expanding the dataset on the fly. We will use Kornia library, as this is what TorchGeo uses, and these transforms also work on GPU and operate on batched data so they are fast!

We will also use a pre-prepared change mask instaed of stacking togehter atasets but that remains a valid option for your usecases.

In [ ]:
import kornia.augmentation as K

# sequential as we have 2 images. The operation is (x - mean) / std, so think about what mean and std
# get our 0-255 images into [0, 1] (hint: K.AugmentationSequential with K.Normalize, data_keys=None, keepdim=True)
normalize = ...

class Mask(RasterDataset):
    is_image = False


# 2018 and 2024 orthophotos with our normalize transform, and change mask from labels/change.tif
before = ...
after = ...
change = ...
# combine, like before
pair = ...

# A land-cover model

In intro we've seen how a regular training and inference loop looks like in basic Torch. TorchGeo is based on Torch Lightning, so it simplifies our life a bit, but still enables all the tinkering.

So to train and use a semantic segmentation model (which assigns a land cover class to each pixel) we use a `SematnicSegmentation` task. It combines model, loss, metrics and optimisation into a neat little block. Here I'd also point out that this is very useful for collaboration as it enables easy testing if stuff works).

I trained the UNet within the SemSeg taaks on LandCover.ai datasets beforehand, as it takes some time. Later we will just adjust it slightly for our data but let's first see how it works.

In [ ]:
# from torchgeo.models import ResNet50_Weights
# landcover = SemanticSegmentation(
#     model='unet', backbone='resnet50', weights=ResNet50_Weights.FMOW_RGB_GASSL, num_classes=5
# )
# load the trained task from checkpoints/ckpt_landcover.ckpt, move it to DEVICE and set it to eval mode
landcover = ...

CLASSES = ListedColormap(['lightgrey', 'red', 'green', 'blue', 'black'])  # background, building, woodland, water, road


def predict(model, image):
    """Classify pixels in image"""
    # run the model on DEVICE, the class is the one with the highest score
    # over the class dimension (dim=1), and bring the result back to cpu
    ...


image = after[458700:458956, 101100:101356]['image']
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image.permute(1, 2, 0))
axes[1].imshow(predict(landcover, image[None])[0], cmap=CLASSES, vmin=0, vmax=4)
axes[1].set_title('red: building, green: woodland, blue: water, black: road')

## Buildings in both years

We only want the building class (1). And we now apply this to our data where we know there are some new houses.

In [ ]:
def buildings(model, image):
    """Get only classified buidlings in image"""
    # use predict from above and keep only the building class
    ...


def show_change(model, x0, y0, size=128):
    sample = pair[x0 : x0 + size, y0 : y0 + size]
    image = sample['image'][None]
    buildings_before = buildings(model, image[:, :3])[0]
    buildings_after = buildings(model, image[:, 3:])[0]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes[0, 0].imshow(image[0, :3].permute(1, 2, 0))
    axes[0, 1].imshow(buildings_before, cmap='gray')
    axes[1, 0].imshow(image[0, 3:].permute(1, 2, 0))
    axes[1, 1].imshow(buildings_after, cmap='gray')
    axes[0, 2].imshow(buildings_before != buildings_after, cmap='gray')
    axes[0, 2].set_title('Predicted change')
    axes[1, 2].imshow(sample['mask'], cmap='gray')
    axes[1, 2].set_title('Cadastre')


show_change(landcover, 458400, 100600)

## How good is the change map?

Okay so we have our segmentations now and with that change maps, but how good are they? We can't really rigorously evaluate that by just looking at maps, so we will use some metrics.

It's a good practice to use reliable implementations, like the one from torchmetrics, so we do not have any mistakes in the implementation.

We will use two metrics here:
- F1: balances missed changes and false alarms.
- IoU (intersection over union): overlap of predicted and true changes.

Both range from 0 to 1, higher is better.

In [ ]:
# here we construct a dataloader from the dataset we had before and a sampler.
# since this is evaluation, we use gridded sampler to cover the entire test region (256 px patches, batch size 8)
test_loader = ...

def score(model):
    f1, iou = BinaryF1Score(), BinaryJaccardIndex()
    for batch in test_loader:
        image = batch['image']
        # first 3 channels are 2018, last 3 are 2024. Predicted change is where the building masks differ
        predicted = ...
        # update both metrics with the prediction and the true change mask
        ...
    print(f'F1: {f1.compute():.2f}, IoU: {iou.compute():.2f}')

score(landcover)

# so it is not great :)

# Fine-tuning on our data

Many deep learning models (and even classic ones) are quite sensitive to the distribution that we train and use them on. So swaping a geographical location from LandCover.ai (Poland) to Slovenia is not necessarily the best idea. Usually to get best performance, we need to perform some "adjsutments" to the model, and we do that by fine-tunign on our data. Since we have a trained model and further tune it, this is also often referred to as transfer learning.

This is usually done on a GPU, you can just skip this step as it takes some time anyway.

In [ ]:
# 2018 orthophoto intersected with 2018 buildings (same as in the datasets notebook)
dataset = ...
# random patches from the train region (256 px, 800 of them) and gridded ones over the val region, batch size 8
train_loader = ...
val_loader = ...
callbacks = [TQDMProgressBar(refresh_rate=1)]
# a new SemanticSegmentation task: unet with resnet50 backbone, but only 2 classes this time (background, building)
task = ...
# only load up to final block (since we go from 5 classes to 2)
state_dict = torch.load('../checkpoints/ckpt_landcover.ckpt', map_location='cpu')['state_dict']
task.load_state_dict({k: v for k, v in state_dict.items() if 'segmentation_head' not in k}, strict=False)

A pretty useful flag for trainer is  `fast_dev_run`, which will run only a single batch of training and validation. This is useful for debugging and testing the setup.

In [ ]:
# Trainer with 10 epochs, no logger, no checkpointing and fast_dev_run on, then fit our task
trainer = ...
...

To avoid training the entire thing, we will load the model that's already trained and repeat the eval above.

Much better, but still not quite great. Good change detectors in most cases reach F1 of 0.8+.
There is always some ambiguity with changes and annotation errors are quick to happen, but what we are seeing here is just not good.

In [ ]:
# load the finetuned task from checkpoints/unet_buildings.ckpt, same as with landcover above
finetuned = ...

show_change(finetuned, 458400, 100600)
score(finetuned)

# Why is this not the best approach?
Every mistake of the segmentation model becomes a change:

A small error in edge segmentation shows as change. Shadows, trees all of that causes potentially unwanted changes. Errors from both years add up, so in next notebook we will see how we can use both images to learn change detection directly.